Python offers three models for doing more than one thing at a time: **threading** for I/O-bound work that spends time waiting, **multiprocessing** for CPU-bound work that needs true parallelism, and **asyncio** for high-concurrency I/O using cooperative coroutines. Knowing which tool fits which problem — and why the GIL makes the choice matter — is the core skill this notebook builds.

## The Global Interpreter Lock (GIL)

CPython has a **Global Interpreter Lock** — a mutex that only one thread can hold at a time. The GIL exists to protect Python's internal reference-counting memory manager.

**Consequence:**
- Multiple threads in the same process share memory but only one executes Python bytecode at a time.
- For **CPU-bound** work (number crunching, image processing), threads do not speed things up — the GIL serialises them.
- For **I/O-bound** work (network calls, file reads, database queries), threads are highly effective — a thread releases the GIL while waiting on I/O, letting other threads run.

The `multiprocessing` module sidesteps the GIL entirely by spawning separate processes, each with its own interpreter and GIL.

`asyncio` uses a single thread with cooperative multitasking — no GIL contention at all.

## Threading — `threading.Thread`

A `Thread` object wraps a callable and runs it in a separate OS thread. Threads share the process's memory space — convenient for sharing data, but requires care to avoid race conditions.

In [ ]:
import threading
import time

def worker(name, delay):
    print(f"[{name}] starting")
    time.sleep(delay)           # simulates I/O wait; releases the GIL
    print(f"[{name}] done after {delay}s")

# Sequential — takes 1 + 2 + 1.5 = 4.5 s
start = time.perf_counter()
for name, delay in [("A", 1), ("B", 2), ("C", 1.5)]:
    worker(name, delay)
print(f"Sequential: {time.perf_counter() - start:.2f}s\n")

# Threaded — all run concurrently; total ≈ longest (2 s)
start = time.perf_counter()
threads = [
    threading.Thread(target=worker, args=(name, delay))
    for name, delay in [("A", 1), ("B", 2), ("C", 1.5)]
]
for t in threads:
    t.start()
for t in threads:
    t.join()                    # wait for all threads to finish
print(f"Threaded:   {time.perf_counter() - start:.2f}s")

In [ ]:
# ThreadPoolExecutor — higher-level API, manages a pool of threads
from concurrent.futures import ThreadPoolExecutor, as_completed
import urllib.request

def fetch_size(url):
    """Return (url, content-length) — simulated here with sleep."""
    time.sleep(0.5)             # simulate network latency
    return url, 1024            # pretend every page is 1 KB

urls = [
    "https://example.com/page1",
    "https://example.com/page2",
    "https://example.com/page3",
    "https://example.com/page4",
]

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(fetch_size, url): url for url in urls}
    for future in as_completed(futures):
        url, size = future.result()
        print(f"{url}: {size} bytes")
print(f"Total: {time.perf_counter() - start:.2f}s")

## Race Conditions and Locks

When multiple threads read and write shared data, the order of operations is non-deterministic. A **race condition** occurs when the result depends on which thread runs first.

The fix is a `Lock` (or `RLock` for re-entrant code) — only one thread can hold the lock at a time.

In [ ]:
# Race condition — counter ends up wrong
counter = 0

def increment_unsafe(n):
    global counter
    for _ in range(n):
        counter += 1    # read-modify-write: not atomic

threads = [threading.Thread(target=increment_unsafe, args=(10_000,)) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Unsafe counter: {counter} (expected 50000)")  # likely < 50000

# Fixed with a Lock
counter = 0
lock = threading.Lock()

def increment_safe(n):
    global counter
    for _ in range(n):
        with lock:         # acquire on entry, release on exit
            counter += 1

threads = [threading.Thread(target=increment_safe, args=(10_000,)) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Safe counter:   {counter} (expected 50000)")

In [ ]:
# threading.Event — signal between threads
ready = threading.Event()

def producer():
    print("Producer: preparing data...")
    time.sleep(1)
    ready.set()             # signal consumers
    print("Producer: data ready")

def consumer(name):
    print(f"Consumer {name}: waiting...")
    ready.wait()            # block until set()
    print(f"Consumer {name}: processing data")

t_prod = threading.Thread(target=producer)
t_cons = [threading.Thread(target=consumer, args=(i,)) for i in range(3)]

for t in t_cons: t.start()
t_prod.start()
t_prod.join()
for t in t_cons: t.join()

In [ ]:
# queue.Queue — thread-safe producer/consumer communication
import queue

def producer_q(q, items):
    for item in items:
        time.sleep(0.1)
        q.put(item)
        print(f"Produced: {item}")
    q.put(None)             # sentinel to signal done

def consumer_q(q):
    while True:
        item = q.get()      # blocks until an item is available
        if item is None:
            break
        print(f"  Consumed: {item}")
        q.task_done()

q = queue.Queue(maxsize=3)  # buffer of at most 3 items
t1 = threading.Thread(target=producer_q, args=(q, range(1, 7)))
t2 = threading.Thread(target=consumer_q, args=(q,))
t1.start(); t2.start()
t1.join();  t2.join()

## Thread-local Storage

`threading.local()` creates storage that is private to each thread — useful for per-thread state such as database connections or request context.

In [ ]:
local_data = threading.local()

def thread_task(name):
    local_data.value = name     # each thread sets its own .value
    time.sleep(0.05)
    print(f"Thread {name}: local_data.value = {local_data.value}")

threads = [threading.Thread(target=thread_task, args=(n,)) for n in ["A", "B", "C"]]
for t in threads: t.start()
for t in threads: t.join()

## Multiprocessing — True Parallelism

For CPU-bound work, use `multiprocessing`. Each worker is a separate OS process with its own GIL, so all CPU cores can be used simultaneously.

Data is passed between processes via serialisation (pickle), which has overhead — multiprocessing pays off only when the computation per task is large enough to outweigh the IPC cost.

In [ ]:
from multiprocessing import Pool
import os

def cpu_bound(n):
    """Sum squares up to n — purely CPU work."""
    return sum(i * i for i in range(n))

if __name__ == "__main__":          # required guard on Windows / macOS
    numbers = [5_000_000] * 4

    # Sequential
    start = time.perf_counter()
    results = [cpu_bound(n) for n in numbers]
    print(f"Sequential: {time.perf_counter() - start:.2f}s")

    # Parallel — uses all available CPU cores
    start = time.perf_counter()
    with Pool() as pool:
        results = pool.map(cpu_bound, numbers)
    print(f"Parallel:   {time.perf_counter() - start:.2f}s")
    print(f"CPU count: {os.cpu_count()}")

In [ ]:
# ProcessPoolExecutor — same interface as ThreadPoolExecutor
from concurrent.futures import ProcessPoolExecutor

def is_prime(n):
    if n < 2: return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

if __name__ == "__main__":
    candidates = [999_999_937, 999_999_929, 999_999_893, 999_999_883]

    with ProcessPoolExecutor() as executor:
        results = list(executor.map(is_prime, candidates))

    for n, prime in zip(candidates, results):
        print(f"{n}: {'prime' if prime else 'not prime'}")

## asyncio — Cooperative Concurrency

`asyncio` runs many tasks on a **single thread** using an **event loop**. Tasks voluntarily yield control at `await` points — when waiting on I/O. This avoids thread-switching overhead and race conditions entirely.

Key vocabulary:
- **coroutine**: a function defined with `async def`. Calling it returns a coroutine object — it does not run until awaited.
- **`await`**: suspends the current coroutine, giving the event loop a chance to run other tasks.
- **`Task`**: a scheduled coroutine running concurrently inside the event loop.
- **event loop**: the scheduler that drives all coroutines.

In [ ]:
import asyncio

async def greet(name, delay):
    print(f"Hello, {name}!")
    await asyncio.sleep(delay)  # yields control — other tasks run during the wait
    print(f"Goodbye, {name}!")

async def main():
    # Run concurrently — total ≈ max(1, 2, 1.5) = 2 s
    await asyncio.gather(
        greet("Alice", 1),
        greet("Bob",   2),
        greet("Carol", 1.5),
    )

# asyncio.run() creates the event loop, runs main(), then closes the loop
import time
start = time.perf_counter()
asyncio.run(main())
print(f"Total: {time.perf_counter() - start:.2f}s")

In [ ]:
# Tasks — schedule coroutines to run concurrently without gather
async def fetch(url, delay):
    await asyncio.sleep(delay)  # simulated I/O
    return f"response from {url}"

async def main():
    # create_task schedules coroutines immediately
    task1 = asyncio.create_task(fetch("https://api.example.com/users",   0.5))
    task2 = asyncio.create_task(fetch("https://api.example.com/orders",  0.8))
    task3 = asyncio.create_task(fetch("https://api.example.com/products", 0.3))

    # await each task in declaration order; all three ran concurrently
    r1 = await task1
    r2 = await task2
    r3 = await task3
    print(r1, r2, r3, sep="\n")

asyncio.run(main())

In [ ]:
# asyncio.gather vs asyncio.wait
async def job(n):
    await asyncio.sleep(n * 0.3)
    if n == 2:
        raise ValueError(f"job {n} failed")
    return f"job {n} done"

async def main():
    # gather — runs all, raises on first exception by default
    try:
        results = await asyncio.gather(job(1), job(2), job(3))
    except ValueError as e:
        print(f"gather raised: {e}")

    # gather with return_exceptions=True — collects all results including errors
    results = await asyncio.gather(job(1), job(2), job(3), return_exceptions=True)
    for r in results:
        print(r)

asyncio.run(main())

## Async Context Managers and Iterators

Resources that require async setup/teardown implement `__aenter__` and `__aexit__`. Sequences that produce values via I/O implement `__aiter__` and `__anext__`.

In [ ]:
class AsyncDB:
    """Simulated async database connection."""

    async def __aenter__(self):
        await asyncio.sleep(0.05)   # simulate connect latency
        print("DB connected")
        return self

    async def __aexit__(self, *exc_info):
        await asyncio.sleep(0.02)   # simulate close
        print("DB disconnected")

    async def query(self, sql):
        await asyncio.sleep(0.1)    # simulate query time
        return [{"id": 1, "sql": sql}]


async def main():
    async with AsyncDB() as db:     # uses __aenter__ / __aexit__
        rows = await db.query("SELECT * FROM users")
        print(rows)

asyncio.run(main())

In [ ]:
# Async generator — yields values with I/O between each
async def async_range(n, delay=0.1):
    for i in range(n):
        await asyncio.sleep(delay)
        yield i

async def main():
    async for value in async_range(5, delay=0.05):  # async for drives the async iterator
        print(value, end=" ")
    print()

    # Async comprehension
    values = [v async for v in async_range(5, delay=0.05)]
    print(values)

asyncio.run(main())

## Timeouts and Cancellation

`asyncio.wait_for` enforces a deadline on a coroutine. A `Task` can be cancelled from outside — a `CancelledError` is injected at its next `await` point, allowing cleanup via `try/finally`.

In [ ]:
async def slow_op():
    print("  starting slow_op")
    await asyncio.sleep(5)
    return "result"

async def main():
    # wait_for — raises asyncio.TimeoutError after the deadline
    try:
        result = await asyncio.wait_for(slow_op(), timeout=1.0)
    except asyncio.TimeoutError:
        print("Timed out!")

asyncio.run(main())

In [ ]:
# Task cancellation
async def long_running():
    try:
        print("Task: running")
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print("Task: cancelled — cleaning up")
        raise           # always re-raise CancelledError
    finally:
        print("Task: finally block")

async def main():
    task = asyncio.create_task(long_running())
    await asyncio.sleep(0.5)    # let task start
    task.cancel()               # request cancellation
    try:
        await task              # wait for it to finish cancelling
    except asyncio.CancelledError:
        print("Main: task is cancelled")

asyncio.run(main())

## asyncio Synchronisation Primitives

asyncio provides coroutine-safe equivalents of threading's primitives — `Lock`, `Event`, `Semaphore`, and `Queue`. Unlike their threading counterparts, they never block the event loop.

In [ ]:
# asyncio.Semaphore — limit concurrent operations (e.g. max open connections)
async def limited_fetch(sem, url, delay):
    async with sem:             # at most 2 concurrent
        await asyncio.sleep(delay)
        return f"fetched {url}"

async def main():
    sem = asyncio.Semaphore(2)  # allow at most 2 concurrent tasks
    urls = [f"url-{i}" for i in range(6)]
    tasks = [limited_fetch(sem, url, 0.3) for url in urls]

    start = time.perf_counter()
    results = await asyncio.gather(*tasks)
    elapsed = time.perf_counter() - start

    for r in results:
        print(r)
    print(f"6 fetches with semaphore(2): {elapsed:.2f}s (expected ≈ 0.9s)")

asyncio.run(main())

In [ ]:
# asyncio.Queue — producer/consumer without threads
async def producer(q, items):
    for item in items:
        await asyncio.sleep(0.1)
        await q.put(item)
        print(f"Produced: {item}")
    await q.put(None)           # sentinel

async def consumer(q, name):
    while True:
        item = await q.get()
        if item is None:
            await q.put(None)   # pass sentinel to other consumers
            break
        print(f"  {name} consumed: {item}")

async def main():
    q = asyncio.Queue(maxsize=3)
    await asyncio.gather(
        producer(q, range(1, 7)),
        consumer(q, "C1"),
    )

asyncio.run(main())

## Running Blocking Code in asyncio

Calling a blocking (synchronous) function directly inside a coroutine stalls the entire event loop. The fix is `loop.run_in_executor()` — it runs the blocking call in a thread pool, returning an awaitable.

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

def blocking_io(name):
    """Pretend this is a legacy synchronous library call."""
    time.sleep(0.5)
    return f"{name}: done"

async def main():
    loop = asyncio.get_running_loop()
    executor = ThreadPoolExecutor(max_workers=4)

    # offload blocking calls to the thread pool; await their results
    tasks = [
        loop.run_in_executor(executor, blocking_io, name)
        for name in ["task-A", "task-B", "task-C"]
    ]

    start = time.perf_counter()
    results = await asyncio.gather(*tasks)
    print(f"Elapsed: {time.perf_counter() - start:.2f}s")
    for r in results:
        print(r)

asyncio.run(main())

## Choosing the Right Model

| Scenario | Tool | Why |
|---|---|---|
| Many network / file / DB calls | `asyncio` or `threading` | GIL released during I/O |
| High connection count (>1000) | `asyncio` | Single thread, no context-switch overhead |
| CPU-heavy computation | `multiprocessing` | True parallelism, bypasses GIL |
| Simple background task | `threading.Thread` | Low overhead, easy to reason about |
| Batch CPU work, pool of workers | `ProcessPoolExecutor` | Clean interface, result futures |
| Mixed I/O + blocking libraries | `asyncio` + `run_in_executor` | Offload blocking to thread pool |

**Rule of thumb:**
- Waiting on the network or disk → `asyncio` (if the library supports it) or `threading`.
- Burning CPU → `multiprocessing`.
- Calling a legacy blocking library from async code → `run_in_executor`.

## Summary

- The **GIL** allows only one thread to execute Python bytecode at a time. Threads are effective for I/O-bound work (the GIL is released during waits) but not for CPU-bound work.
- `threading.Thread` and `ThreadPoolExecutor` run callables in OS threads sharing the same memory. Use `Lock`, `Event`, and `queue.Queue` to coordinate safely.
- `multiprocessing.Pool` and `ProcessPoolExecutor` spawn separate processes — each with their own GIL — giving true CPU parallelism. Data passes between processes via pickle.
- `asyncio` uses a single-threaded **event loop** with cooperative multitasking. Coroutines (`async def`) yield at `await` points, letting other tasks run during I/O waits. No race conditions, minimal overhead.
- `asyncio.gather` runs coroutines concurrently and collects results. `create_task` schedules a coroutine immediately without awaiting it.
- `async with` and `async for` work with resources and generators that perform async operations.
- `asyncio.wait_for` enforces timeouts. `task.cancel()` injects `CancelledError` at the next `await`.
- `asyncio.Semaphore` limits concurrent operations. `asyncio.Queue` provides safe producer/consumer communication.
- Use `loop.run_in_executor()` to call blocking synchronous code from within an asyncio coroutine without stalling the event loop.